# EIA-860 Renewable Site Merge, Filter, and Regrid Methodology

This notebook shows how EIA-860 renewable generator records are converted into wind and solar grid points (`gid`) used to query NLR weather datasets, including WTK, BC-HRRR, NSRDB, and Sup3rCC. 

The notebook produces a csv for each BA and weather model. Each row contains a resource-grid gid and the total installed capacity assigned to that grid point.

Downstream of this notebook, `site_cf_generation_ba_weighting_validation` downloads weather data at the grid ids from the respective weather model, converts that weather data to a wind or solar capacity factor using `rev`, and weights those capacity factors by the installed capacity to get the weighted capacity factor for a certain BA. 

## Imports And Package Setup

Load the Python libraries used below and define the package-relative roots used by the workflow.


In [1]:
from pathlib import Path

import pandas as pd
from rex import Resource
from scipy.spatial import cKDTree

DATA_ROOT = Path("../../data")
OUTPUT_DIR = Path("../../notebook_outputs/data_flow/eia860_regridding_methodology")

if not DATA_ROOT.is_dir():
    raise FileNotFoundError(
        "Expected to find the packaged data folder at ../data. "
        "Open this notebook from inside notebooks/data_flow."
    )

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## Step 1: Read EIA-860 Workbooks

The plant workbook supplies plant location and balancing-authority fields. The generator workbook supplies generator status, technology, generator ID, and nameplate capacity. Because `STATUS = "Operable"`, this notebook reads the `Operable` sheet from the generator workbook. 

`Operable` refers to generators that are currently in service (as opposed to `Proposed`).

In [2]:
YEAR = 2024
STATUS = "Operable"

PLANT_XLSX = DATA_ROOT / "eia860" / "2024_raw" / "2___Plant_Y2024.xlsx"
GENERATOR_XLSX = DATA_ROOT / "eia860" / "2024_raw" / "3_1_Generator_Y2024.xlsx"

plant = pd.read_excel(PLANT_XLSX, sheet_name="Plant", header=1)
generator = pd.read_excel(GENERATOR_XLSX, sheet_name=STATUS, header=1)

display(pd.DataFrame([
    {"table": "plant", "workbook": PLANT_XLSX.name, "sheet": "Plant", "rows": len(plant), "columns": len(plant.columns)},
    {"table": "generator", "workbook": GENERATOR_XLSX.name, "sheet": STATUS, "rows": len(generator), "columns": len(generator.columns)},
]))

plant_columns = pd.DataFrame({"plant_columns": plant.columns})
generator_columns = pd.DataFrame({"generator_columns": generator.columns})

#with pd.option_context("display.max_rows", None):
display(pd.DataFrame({"plant_columns": plant.columns}))
display(pd.DataFrame({"generator_columns": generator.columns}))

,table,workbook,sheet,rows,columns
0,plant,2___Plant_Y2024.xlsx,Plant,16132,42
1,generator,3_1_Generator_Y2024.xlsx,Operable,26856,73


,plant_columns
0,Utility ID
1,Utility Name
2,Plant Code
3,Plant Name
4,Street Address
5,City
6,State
7,Zip
8,County
9,Latitude


,generator_columns
0,Utility ID
1,Utility Name
2,Plant Code
3,Plant Name
4,State
...,...
68,Other Modifications Month
69,Other Modifications Year
70,Multiple Fuels?
71,Cofire Fuels?


## Step 2: Add Plant Fields to Generator Rows

Merge plant coordinates and balancing-authority fields onto each generator. Keep the five fields needed for filtering and aggregation, plus plant code, plant name, and generator ID for readable examples.

In [3]:
MERGE_KEYS = [
    "Utility ID",
    "Utility Name",
    "Plant Code",
    "Plant Name",
    "State",
    "County",
    "Sector",
    "Sector Name",
]

MERGED_COLUMNS_TO_KEEP = [
    "Plant Code",
    "Plant Name",
    "Generator ID",
    "Technology",
    "Nameplate Capacity (MW)",
    "Latitude",
    "Longitude",
    "Balancing Authority Code",
]

# Attach plant fields to each generator row.
merged_raw = generator.merge(plant, on=MERGE_KEYS, how="left", indicator=True)
merged = merged_raw[MERGED_COLUMNS_TO_KEEP].copy()

# Clean the columns used in later filters and capacity sums.
merged["Balancing Authority Code"] = merged["Balancing Authority Code"].astype(str).str.strip().str.upper()
merged["Nameplate Capacity (MW)"] = pd.to_numeric(merged["Nameplate Capacity (MW)"])
merged["Latitude"] = pd.to_numeric(merged["Latitude"])
merged["Longitude"] = pd.to_numeric(merged["Longitude"])

merge_summary = pd.DataFrame([
    {"check": "generator rows", "value": len(generator)},
    {"check": "merged rows", "value": len(merged_raw)},
    {"check": "unmatched generator rows", "value": int((merged_raw["_merge"] != "both").sum())},
    {"check": "merged rows missing lat/lon", "value": int(merged[["Latitude", "Longitude"]].isna().any(axis=1).sum())},
])

display(merge_summary)

display(merged[MERGED_COLUMNS_TO_KEEP].head(50))

,check,value
0,generator rows,26856
1,merged rows,26856
2,unmatched generator rows,2
3,merged rows missing lat/lon,2


,Plant Code,Plant Name,Generator ID,Technology,Nameplate Capacity (MW),Latitude,Longitude,Balancing Authority Code
0,1.0,Sand Point,1,Petroleum Liquids,0.9,55.339722,-160.497222,NAN
1,1.0,Sand Point,2,Petroleum Liquids,0.9,55.339722,-160.497222,NAN
2,1.0,Sand Point,3,Petroleum Liquids,0.5,55.339722,-160.497222,NAN
3,1.0,Sand Point,5.1,Petroleum Liquids,0.4,55.339722,-160.497222,NAN
4,1.0,Sand Point,WT1,Onshore Wind Turbine,0.5,55.339722,-160.497222,NAN
5,1.0,Sand Point,WT2,Onshore Wind Turbine,0.5,55.339722,-160.497222,NAN
6,2.0,Bankhead Dam,1,Conventional Hydroelectric,53.9,33.458665,-87.356823,SOCO
7,3.0,Barry,1,Natural Gas Steam Turbine,153.1,31.006900,-88.010300,SOCO
8,3.0,Barry,2,Natural Gas Steam Turbine,153.1,31.006900,-88.010300,SOCO
9,3.0,Barry,4,Conventional Steam Coal,403.7,31.006900,-88.010300,SOCO


## Step 3: Filter To Wind/Solar For One BA

This step starts with the merged EIA-860 generator table, keeps only wind and solar generator rows, and then keeps only rows assigned to the example balancing authority. 

The filtered table, `ba_renewable`, is used in Step 4.

In [4]:
BA_CODE = "MISO"
RENEWABLE_TECHNOLOGIES = ["Onshore Wind Turbine", "Solar Photovoltaic"]

# Keep only the renewable generator technologies used by the wind/solar CF workflow.
renewable = merged[merged["Technology"].isin(RENEWABLE_TECHNOLOGIES)].copy()

# Keep only renewable generator rows assigned to the selected BA.
ba_renewable = renewable[renewable["Balancing Authority Code"] == BA_CODE].copy()

filter_summary = pd.DataFrame([
    {"check": "all generator rows", "value": len(merged)},
    {"check": "renewable generator rows", "value": len(renewable)},
    {"check": f"{BA_CODE} renewable generator rows", "value": len(ba_renewable)},
    {
        "check": f"{BA_CODE} renewable rows missing lat/lon",
        "value": int(ba_renewable[["Latitude", "Longitude"]].isna().any(axis=1).sum()),
    },
])

display(filter_summary)


,check,value
0,all generator rows,26856
1,renewable generator rows,8702
2,MISO renewable generator rows,1579
3,MISO renewable rows missing lat/lon,0


## Step 4: Aggregate Generator Rows To Site Coordinates

This step combines aggregates generators that share the same `Technology`, `Latitude`, and `Longitude` for the `BA_CODE` (filtered above, in step 3) into one site row, as multiple generators may share the same plant lat/lon.

The first cell shows one real example: multiple generator rows at the same technology and plant coordinates before aggregation, followed by the single site row created after their `Nameplate Capacity (MW)` values are summed.

The second cell applies the same aggregation to all rows in `ba_renewable`. Wind and solar remain separate because `Technology` is included in the grouping.

In [5]:
SITE_GROUP_COLUMNS = ["Technology", "Latitude", "Longitude"]

with_coordinates = ba_renewable.dropna(subset=["Latitude", "Longitude"])

site_row_counts = with_coordinates.groupby(SITE_GROUP_COLUMNS).size().reset_index(name="generator_rows")
duplicate_site = site_row_counts[site_row_counts["generator_rows"] > 1].iloc[0]

same_site = (
    (with_coordinates["Technology"] == duplicate_site["Technology"])
    & (with_coordinates["Latitude"] == duplicate_site["Latitude"])
    & (with_coordinates["Longitude"] == duplicate_site["Longitude"])
)

before = with_coordinates[same_site]
print("Before aggregation: generator rows at the same technology and lat/lon")
display(before[MERGED_COLUMNS_TO_KEEP])

after = before.groupby(SITE_GROUP_COLUMNS, as_index=False).agg(site_nameplate_mw=("Nameplate Capacity (MW)", "sum"))
print("After aggregation: one site row with summed nameplate capacity")
display(after)

Before aggregation: generator rows at the same technology and lat/lon


,Plant Code,Plant Name,Generator ID,Technology,Nameplate Capacity (MW),Latitude,Longitude,Balancing Authority Code
17748,59637.0,Adams Wind,ADWF,Onshore Wind Turbine,4.7,40.92,-94.671667,MISO
17749,59637.0,Adams Wind,ADWF2,Onshore Wind Turbine,43.3,40.92,-94.671667,MISO
17750,59637.0,Adams Wind,ADWF3,Onshore Wind Turbine,58.0,40.92,-94.671667,MISO
17751,59637.0,Adams Wind,ADWF4,Onshore Wind Turbine,48.3,40.92,-94.671667,MISO


After aggregation: one site row with summed nameplate capacity


,Technology,Latitude,Longitude,site_nameplate_mw
0,Onshore Wind Turbine,40.92,-94.671667,154.3


In [6]:
ba_site_capacity = with_coordinates.groupby(SITE_GROUP_COLUMNS, as_index=False).agg(site_nameplate_mw=("Nameplate Capacity (MW)", "sum"))

generator_summary = ba_renewable.groupby("Technology", as_index=False).agg(generator_rows=("Technology", "size"))

site_summary = ba_site_capacity.groupby("Technology", as_index=False).agg(unique_site_coordinates=("site_nameplate_mw", "size"), site_nameplate_mw=("site_nameplate_mw", "sum"), )

ba_site_summary = generator_summary.merge(site_summary, on="Technology")

wind_site_capacity = ba_site_capacity.query("Technology == 'Onshore Wind Turbine'").copy()
solar_site_capacity = ba_site_capacity.query("Technology == 'Solar Photovoltaic'").copy()

print(f"{BA_CODE} renewable generator rows aggregated to site coordinates")
display(ba_site_summary)

MISO renewable generator rows aggregated to site coordinates


,Technology,generator_rows,unique_site_coordinates,site_nameplate_mw
0,Onshore Wind Turbine,427,356,32150.7
1,Solar Photovoltaic,1152,954,13574.9


## Step 5: Match BA Site Coordinates To HSDS Grid Points

This step maps renewable site coordinates to the WTK, BC-HRRR, and NSRDB grid IDs used by the site-weather workflow. Rows sharing a selected gid are grouped and their EIA-860 nameplate capacity is summed.

Before running this step, start the local HSDS service at `http://localhost:5101` with access to the `nrel-pds-hsds` bucket. The cKDTree reconstruction runs directly, and the following checks compare its selected grid points with the reviewed packaged CSVs.


In [7]:
HSDS_ENDPOINT = "http://localhost:5101"
HSDS_API_KEY = None
HSDS_BUCKET = "nrel-pds-hsds"

# Use one representative resource year for each source's historical grid.
RESOURCE_PATHS = {
    "wtk": "/nrel/wtk/conus/wtk_conus_2013.h5",
    "bchrrr": "/nrel/wtk/bchrrr/v1.0.0/bchrrr_conus_2019.h5",
    "nsrdb": "/nrel/nsrdb/GOES/aggregated/v4.0.0/nsrdb_2019.h5",
}
BA_REGRIDDED_POINTS_CSV = OUTPUT_DIR / f"{BA_CODE}_{YEAR}_regridded_hsds_grid_points.csv"

site_capacity_by_source = {
    "wtk": wind_site_capacity,
    "bchrrr": wind_site_capacity,
    "nsrdb": solar_site_capacity,
}
regridded_tables = []
for weather_source, resource_path in RESOURCE_PATHS.items(): #This shares symmetry with county_weather_point_selection.ipynb
    # Select the wind or solar sites for this weather source.
    sites = site_capacity_by_source[weather_source]

    # 1. Read the latitude/longitude grid for this weather source.
    with Resource(
        resource_path,
        hsds=True,
        hsds_kwargs={
            "endpoint": HSDS_ENDPOINT,
            "api_key": HSDS_API_KEY,
            "bucket": HSDS_BUCKET,
        },
    ) as resource:
        grid_coordinates = resource.coordinates

    # 2. The coordinate row number is the HSDS grid id.
    tree = cKDTree(grid_coordinates)
    _, selected_gids = tree.query(
        sites[["Latitude", "Longitude"]].to_numpy(dtype=float)
    )
    selected_coordinates = grid_coordinates[selected_gids]

    # 3. Store the selected grid ids and coordinates for this weather source.
    matched = sites.assign(
        weather_source=weather_source,
        selected_gid=selected_gids,
        grid_lat=selected_coordinates[:, 0],
        grid_lon=selected_coordinates[:, 1],
    )

    # 4. Sum capacity when multiple sites map to the same gid.
    regridded_tables.append(
        matched.groupby(
            ["weather_source", "Technology", "selected_gid", "grid_lat", "grid_lon"],
            as_index=False,
        )["site_nameplate_mw"].sum()
    )

# Combine the results and write the reconstructed grid-point file.
nearest_grid_points = pd.concat(regridded_tables, ignore_index=True)
nearest_grid_points.to_csv(BA_REGRIDDED_POINTS_CSV, index=False)

print(f"Wrote HSDS reconstruction: {BA_REGRIDDED_POINTS_CSV.relative_to(Path('../..'))}")

# Summarize the grid-point count and total capacity by weather source.
display(nearest_grid_points.groupby("weather_source", as_index=False).agg(
        grid_points=("selected_gid", "nunique"),
        nameplate_mw=("site_nameplate_mw", "sum"),
    )
)

Wrote HSDS reconstruction: notebook_outputs\data_flow\eia860_regridding_methodology\MISO_2024_regridded_hsds_grid_points.csv


,weather_source,grid_points,nameplate_mw
0,bchrrr,317,32150.7
1,nsrdb,721,13574.9
2,wtk,317,32150.7


## Appendix: MISO Renewable Subregion Mapping

For MISO renewables, the subregion split happens only at the weighted-CF step:

- `MISO_wtk.csv`, `MISO_bchrrr.csv`, and `MISO_nsrdb.csv` are total-MISO regridded reference files.
- Site-weather H5 files are total-MISO files.
- Site-CF H5 files are total-MISO files.
- The spreadsheet `eia860_2024_operable_miso_subregions.xlsx` maps each total-MISO renewable `gid` to `miso_lrz`, `ba_code`, and `ba_number`.
- `ba_code` identifies the MISO renewable subregion each row belongs to, such as `MISO_0001`.
- The weighted BA-level CF step filters the total-MISO site-CF H5 by `ba_code` and writes both total-MISO and MISO subregion CF CSVs.

The code below reads the packaged spreadsheet and summarizes how many total-MISO grid IDs are assigned to each MISO renewable subregion for each weather source.



In [8]:
MISO_SUBREGION_SPREADSHEET = (DATA_ROOT / "eia860" / "2024_regridded_points" / "eia860_2024_operable_miso_subregions.xlsx")

print(f"MISO renewable subregion spreadsheet: {MISO_SUBREGION_SPREADSHEET.relative_to(Path('../..'))}")

miso_ba_map = pd.read_excel(MISO_SUBREGION_SPREADSHEET, sheet_name="mapping", dtype=str).fillna("")
miso_ba_map["gid"] = pd.to_numeric(miso_ba_map["gid"], errors="coerce")
miso_ba_summary = miso_ba_map.groupby(["source_key", "ba_code", "ba_number"], as_index=False).agg(rows=("gid", "size"), unique_gids=("gid", "nunique")).sort_values(["source_key", "ba_code", "ba_number"])

preview_columns = ["generator_set", "source_key", "gid", "lat", "lon", "miso_lrz", "ba_code", "ba_number", "assignment_method"]

display(miso_ba_summary)
display(miso_ba_map[preview_columns].head(10))

MISO renewable subregion spreadsheet: data\eia860\2024_regridded_points\eia860_2024_operable_miso_subregions.xlsx


,source_key,ba_code,ba_number,rows,unique_gids
0,bchrrr,MISO_0001,0001,133,133
1,bchrrr,MISO_0004,0004,23,23
2,bchrrr,MISO_0006,0006,7,7
3,bchrrr,MISO_0027,0027,42,42
4,bchrrr,MISO_0035,0035,111,111
...,...,...,...,...,...
181,wtk,MISO_0004,0004,23,23
182,wtk,MISO_0006,0006,7,7
183,wtk,MISO_0027,0027,42,42
184,wtk,MISO_0035,0035,111,111


,generator_set,source_key,gid,lat,lon,miso_lrz,ba_code,ba_number,assignment_method
0,eia860_2024_operable,bchrrr,936134,46.277283,-104.19415,LRZ1,MISO_0001,0001,shapefile_overlay
1,eia860_2024_operable,bchrrr,952844,46.251312,-103.75714,LRZ1,MISO_0001,0001,shapefile_overlay
2,eia860_2024_operable,bchrrr,998019,46.0817,-102.57388,LRZ1,MISO_0001,0001,shapefile_overlay
3,eia860_2024_operable,bchrrr,1032943,46.963448,-101.80423,LRZ1,MISO_0001,0001,shapefile_overlay
4,eia860_2024_operable,bchrrr,1043135,46.975338,-101.559204,LRZ1,MISO_0001,0001,shapefile_overlay
5,eia860_2024_operable,bchrrr,1059181,46.95563,-101.17435,LRZ1,MISO_0001,0001,shapefile_overlay
6,eia860_2024_operable,bchrrr,1059192,47.160065,-101.19379,LRZ1,MISO_0001,0001,shapefile_overlay
7,eia860_2024_operable,bchrrr,1072015,47.860577,-100.95578,LRZ1,MISO_0001,0001,shapefile_overlay
8,eia860_2024_operable,bchrrr,1074361,48.08582,-100.92084,LRZ1,MISO_0001,0001,shapefile_overlay
9,eia860_2024_operable,bchrrr,1079037,47.1239,-100.72415,LRZ1,MISO_0001,0001,shapefile_overlay


## Downstream Use

The next notebook, `site_cf_generation_ba_weighting_validation.ipynb`, shows how selected `gid` values are used to read download (wind and solar) site-level weather data, create site-level CF profiles, and aggregate those profiles with EIA-860 nameplate capacity weights.